# Day 1: Website Summarizer with Local Ollama

This notebook follows the `llm_engineering/week1/day1.ipynb` workflow: fetch a webpage, build system/user messages, ask a chat model for a summary, and render the result. It uses the local Ollama settings in this folder's `.env`; no cloud API key is needed.

Run cells from the top. If the first dependency check fails, run the install cell and restart the kernel. Ollama must be running locally and the configured model must already be available. The example URLs make external HTTP requests.

## 1. Set Up Environment and Dependencies

Run the next cell once in the notebook's selected Python kernel. It installs only the libraries used by this lab; restart the kernel if the notebook asks you to.

In [1]:
%pip install -q "python-dotenv>=1.0" "requests>=2.31" "beautifulsoup4>=4.12" "openai>=1.0"

/Users/Apple/Projects/llm_engineering/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## 2. Load `.env` Configuration for Local Ollama

The configuration cell requires `OLLAMA_BASE_URL` and `OLLAMA_MODEL`. Optional `OLLAMA_TIMEOUT_SECONDS`, `OLLAMA_TEMPERATURE`, `OLLAMA_MAX_TOKENS`, `OLLAMA_TOP_P`, `OLLAMA_SEED`, `OLLAMA_API_KEY`, and `OLLAMA_MAX_RETRIES` values have safe defaults. It reports presence only and never prints credentials.

In [15]:
from importlib.metadata import PackageNotFoundError, version
import json
import os
from pathlib import Path
import time
from urllib.parse import urlsplit

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
import openai
import requests

ENV_PATH = Path.cwd() / ".env"
if not ENV_PATH.is_file():
    raise FileNotFoundError(f"Expected .env beside this notebook: {ENV_PATH}")

load_dotenv(ENV_PATH, override=True)

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "").strip().rstrip("/")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "").strip()
if not OLLAMA_BASE_URL or not OLLAMA_MODEL:
    raise ValueError(".env must define OLLAMA_BASE_URL and OLLAMA_MODEL")

if not OLLAMA_BASE_URL.endswith("/v1"):
    OLLAMA_BASE_URL += "/v1"
OLLAMA_SERVER_URL = OLLAMA_BASE_URL[:-3].rstrip("/")
OLLAMA_TIMEOUT_SECONDS = float(os.getenv("OLLAMA_TIMEOUT_SECONDS", "120"))
OLLAMA_TEMPERATURE = float(os.getenv("OLLAMA_TEMPERATURE", "0"))
OLLAMA_MAX_TOKENS = int(os.getenv("OLLAMA_MAX_TOKENS", "1024"))
OLLAMA_TOP_P = float(os.getenv("OLLAMA_TOP_P", "1"))
OLLAMA_SEED = int(os.getenv("OLLAMA_SEED", "42"))
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY", "ollama")
MAX_RETRIES = int(os.getenv("OLLAMA_MAX_RETRIES", "2"))

print(f"Loaded .env: {ENV_PATH.name}")
print(f"Ollama endpoint configured: {urlsplit(OLLAMA_BASE_URL).hostname or 'yes'}")
print("Ollama model configured: yes")
for package_name in ("python-dotenv", "requests", "beautifulsoup4", "openai"):
    try:
        print(f"{package_name}: {version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name}: version unavailable")

Loaded .env: .env
Ollama endpoint configured: localhost
Ollama model configured: yes
python-dotenv: 1.2.1
requests: 2.32.5
beautifulsoup4: 4.14.2
openai: 2.7.1


## 3. Recreate Core Notebook Variables

These prompt variables mirror the original Day 1 summarizer while keeping provider and model configuration local.

In [16]:
SYSTEM_PROMPT = """You are a helpful assistant that summarizes website content accurately.
Ignore navigation and boilerplate. Respond in concise Markdown and do not invent facts."""
USER_PROMPT_PREFIX = """Summarize the following website. Include its purpose and important news or announcements when present.

"""
TEST_MESSAGE = "Hello, Ollama! This is my first local message."
SAMPLE_URL = "https://edwarddonner.com"

## 4. Initialize the Local Ollama Client

Ollama exposes an OpenAI-compatible chat endpoint. The API key below is only a placeholder for local compatibility; no cloud credential is required. Temperature defaults to zero for more repeatable demonstrations.

In [17]:
client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=OLLAMA_API_KEY,
    timeout=OLLAMA_TIMEOUT_SECONDS,
    max_retries=0,
)

MODEL_PARAMETERS = {
    "temperature": OLLAMA_TEMPERATURE,
    "max_tokens": OLLAMA_MAX_TOKENS,
    "top_p": OLLAMA_TOP_P,
    "seed": OLLAMA_SEED,
}
print(f"Client ready for configured local model: {OLLAMA_MODEL}")

Client ready for configured local model: phi3


## 5. Build Prompting and Response Helpers

The scraper is inline so this notebook works from `Practice_llm_engineering` without importing files from the other repository. Its signature and message-building flow stay close to the original lab.

In [18]:
REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; Day1LocalSummarizer/1.0)"
}


def fetch_website_contents(url):
    """Fetch a page and return its title and readable text, capped at 2,000 characters."""
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.get_text(" ", strip=True) if soup.title else "No title found"
    body = soup.body
    if body:
        for irrelevant in body(["script", "style", "img", "input", "noscript"]):
            irrelevant.decompose()
        text = body.get_text(separator="\n", strip=True)
    else:
        text = soup.get_text(separator="\n", strip=True)
    return f"{title}\n\n{text}"[:2_000]


def messages_for(website):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT_PREFIX + website},
    ]

In [19]:
def extract_response_text(completion):
    """Extract assistant text from a non-streaming chat completion."""
    return completion.choices[0].message.content or ""

In [20]:
def summarize(url):
    """Fetch a URL and return its local Ollama summary."""
    website = fetch_website_contents(url)
    return extract_response_text(call_ollama(messages_for(website)))


def display_summary(url):
    """Render a website summary as Markdown in the notebook."""
    display(Markdown(summarize(url)))

## 6. Run a Single-Turn Prompt Example

First send a tiny prompt to confirm the model responds. The check is structural (non-empty text), not an exact-string comparison, because local model outputs can vary.

In [21]:
test_messages = [{"role": "user", "content": TEST_MESSAGE}]
try:
    test_response = client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=test_messages,
        **MODEL_PARAMETERS,
    )
    test_reply = extract_response_text(test_response)
    checks = {
        "reply_is_text": isinstance(test_reply, str),
        "reply_is_non_empty": bool(test_reply.strip()),
    }
    print(test_reply)
    print("Behavior checks:", checks)
    assert all(checks.values()), "The local model returned an empty or invalid reply."
except openai.APIError as error:
    print(f"Ollama request failed ({type(error).__name__}). Run the connectivity checks below.")

Hello! I'm thrilled to be here with you. How can I assist you today?
Behavior checks: {'reply_is_text': True, 'reply_is_non_empty': True}


## 7. Run a Multi-Step Prompt Workflow

This optional extension performs two local calls: first it extracts source-grounded key points, then it summarizes those points. It is useful for comparing a staged workflow with the single-call version.

In [22]:
def summarize_in_stages(url):
    website = fetch_website_contents(url)
    fact_messages = [
        {"role": "system", "content": "Extract only key facts and announcements from the source."},
        {"role": "user", "content": website},
    ]
    key_points = extract_response_text(call_ollama(fact_messages))
    final_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": "Write a concise Markdown summary using these extracted points:\n\n" + key_points,
        },
    ]
    summary = extract_response_text(call_ollama(final_messages))
    return {"url": url, "key_points": key_points, "summary": summary}

## 8. Add Streaming and Response Parsing

Streaming is optional; it prints text as Ollama yields chunks. The JSON helper accepts either plain JSON or a fenced JSON block and raises a clear parse error otherwise.

In [23]:
def stream_ollama(messages):
    stream = client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=messages,
        stream=True,
        **MODEL_PARAMETERS,
    )
    for chunk in stream:
        if chunk.choices:
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece


def parse_json_response(text):
    """Parse a JSON object returned as plain text or inside a Markdown fence."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[-1]
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3].strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        start, end = cleaned.find("{"), cleaned.rfind("}")
        if start < 0 or end <= start:
            raise ValueError("The model response did not contain a JSON object.")
        return json.loads(cleaned[start : end + 1])


def display_streamed_summary(url):
    website = fetch_website_contents(url)
    pieces = []
    for piece in stream_ollama(messages_for(website)):
        pieces.append(piece)
        print(piece, end="", flush=True)
    print()
    return "".join(pieces)

## 9. Add Robust Error Handling and Retries

The client has a bounded request timeout. This helper retries only connection, timeout, rate-limit, and server-side transient errors; configuration/model errors fail immediately.

In [24]:
RETRYABLE_OLLAMA_ERRORS = (
    openai.APIConnectionError,
    openai.APITimeoutError,
    openai.RateLimitError,
    openai.InternalServerError,
)


def call_ollama(messages):
    retries = max(0, MAX_RETRIES)
    for attempt in range(retries + 1):
        try:
            return client.chat.completions.create(
                model=OLLAMA_MODEL,
                messages=messages,
                **MODEL_PARAMETERS,
            )
        except RETRYABLE_OLLAMA_ERRORS as error:
            if attempt >= retries:
                raise
            delay = min(0.5 * (2**attempt), 4.0)
            print(f"Temporary Ollama error ({type(error).__name__}); retrying in {delay:.1f}s.")
            time.sleep(delay)

## 10. Validate Local Ollama Connectivity

This checks the Ollama tags endpoint and whether the configured model is installed. If the server is unreachable, start Ollama from a terminal (for example, `ollama serve`); if the model is missing, pull the configured model with `ollama pull <model>`.

In [25]:
def diagnose_ollama():
    tags_url = f"{OLLAMA_SERVER_URL}/api/tags"
    try:
        response = requests.get(tags_url, timeout=5)
        response.raise_for_status()
        model_names = [
            item.get("name") or item.get("model", "")
            for item in response.json().get("models", [])
        ]
    except (requests.RequestException, ValueError) as error:
        print(f"Ollama server check failed ({type(error).__name__}).")
        print("Start Ollama, then rerun this cell. Check the configured host if it still fails.")
        return {"server_reachable": False, "model_available": False}

    model_available = any(
        name == OLLAMA_MODEL or name.split(":", 1)[0] == OLLAMA_MODEL.split(":", 1)[0]
        for name in model_names
    )
    print("Ollama server reachable: yes")
    print(f"Configured model available: {'yes' if model_available else 'no'}")
    if not model_available:
        print("Pull the configured model from a terminal, then rerun this cell.")
    return {"server_reachable": True, "model_available": model_available}


diagnostics = diagnose_ollama()

Ollama server reachable: yes
Configured model available: yes


In [26]:
if diagnostics["server_reachable"] and diagnostics["model_available"]:
    try:
        summary_text = summarize(SAMPLE_URL)
        display(Markdown(summary_text))
    except (requests.RequestException, openai.APIError) as error:
        summary_text = None
        print(f"Website summary failed ({type(error).__name__}).")
else:
    summary_text = None
    print("Skipping the website example until Ollama and the configured model are available.")

**Edward Donner's Website Summary**

Edward Donner is a co-founder and CTO of AI startup Nebula.io, and he has a background as a founder and CEO of untapt, which was acquired in 2021, and as a Managing Director at JPMorgan. He is passionate about writing code, experimenting with LLMs, amateur electronic music production, and engaging with Hacker News.

Edward has created a series of Udemy courses on LLMs, which have become best-selling, top-rated courses with 900,000 enrollments across 194 countries. The full curriculum is available on his website.

Edward also hosts an arena called "Outsmart," where LLMs compete in a battle of diplomacy and deviousness.

To get in touch with Edward, you can contact him via email at ed [at] edwarddonner [dot] com. He also has a presence on LinkedIn, Twitter, and Facebook.

**Recent News/Announcements**

- February 17, 2026: Edward Donner announced the release of his AI Coder: Vibe Coder to Agentic Engineer course.
- January 4, 2026: Edward Donner announced the release of his AI Builder with n8n course.
- September 15, 2025: Edward Donner announced the release of his AI Engineering MLOps Track course.
- May 28, 2025: Edward Donner announced the release of his AI Courses series.

**Course Order Recommendation**

While the website does not explicitly mention the order to take the AI courses, it is recommended to start with the AI Courses series, which includes the AI Coder: Vibe Coder to Agentic Engineer course, followed by the AI Builder with n8n course, and then the AI Engineering MLOps Track course.

## 11. Persist Outputs and Reproducibility Checks

Temperature zero and a fixed seed make reruns more comparable, though exact determinism depends on the model and Ollama version. Saved artifacts include prompts/results metadata but never environment variables or API credentials.

In [27]:
def save_run_artifact(url, summary, key_points=None, output_dir="artifacts"):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    artifact = {
        "url": url,
        "model": OLLAMA_MODEL,
        "parameters": MODEL_PARAMETERS,
        "summary": summary,
        "key_points": key_points,
    }
    artifact_file = output_path / "day1_summary.json"
    artifact_file.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
    return artifact_file


if summary_text:
    saved_path = save_run_artifact(SAMPLE_URL, summary_text)
    print(f"Saved summary artifact: {saved_path}")
else:
    print("No summary to save yet. Run the website example after Ollama is available.")

Saved summary artifact: artifacts/day1_summary.json


## Try More Websites and Variations

Use the same workflow with pages that can be scraped from their HTML. JavaScript-heavy or access-protected sites may need a browser-based scraper. The staged and streaming helpers are optional alternatives to `display_summary`.

In [29]:
assert [message["role"] for message in messages_for("sample page")] == ["system", "user"]
assert parse_json_response('{"ok": true}') == {"ok": True}
assert parse_json_response('```json\n{"ok": true}\n```') == {"ok": True}
print("Prompt formatting and JSON parsing checks passed.")

# Uncomment to try other sites and workflows:
display_summary("https://cnn.com")
# display_summary("https://anthropic.com")
# staged_result = summarize_in_stages(SAMPLE_URL)
# streamed_result = display_streamed_summary(SAMPLE_URL)

Prompt formatting and JSON parsing checks passed.



**Summary of CNN's Website:**

CNN's website is a news and video platform that provides breaking news, latest news, and videos on various topics such as politics, business, health, entertainment, and more. The website also values user feedback and allows users to provide their opinions on the ads displayed on the site.

**Important News or Announcements:**

The website includes a section for user feedback on ads, where users can rate the relevance of ads, report technical issues, and provide other feedback. The website also features live TV, watch, and listen options, as well as a redistricting tracker and other interactive elements.

**Purpose of the Website:**

The purpose of the website is to provide users with the latest news and videos on various topics, as well as to offer a platform for user feedback and interaction. The website also aims to engage users through live TV, watch, and listen options, and to provide relevant and timely news coverage.

**Ad Feedback:**

The website includes a section for users to provide feedback on the ads displayed on the site, including issues such as ad relevance, technical problems, and other concerns. The website also allows users to submit their feedback and express their opinions on the ads.

**Live TV, Watch, and Listen:**

The website offers live TV, watch, and listen options for users to stay updated with the latest news and events. Users can also sign in to their CNN account to access personalized content and features.

**Redistricting Tracker:**

The website includes a redistricting tracker, which provides information on redistricting efforts and changes in the United States. This feature allows users to stay informed on political developments and changes in their communities.

**Ukraine-Russia War and Israel-Hamas War:**

The website covers the ongoing Ukraine-Russia War and Israel-Hamas War, providing news and updates on these conflicts. The website also includes a section for user feedback on the coverage of these events.

**Ad Feedback:**

The website includes a section for users to provide feedback on the ads displayed on the site, including issues such as ad relevance, technical problems, and other concerns. The website also allows users to submit their feedback and express their opinions on the ads.

**Live TV, Watch, and Listen:**

The website offers live TV, watch, and listen options for users to stay updated with the latest news and events. Users can also sign in to their CNN account to access personalized content and features.

**Redistricting Tracker:**

The website includes a redistricting tracker, which provides information on redistricting efforts and changes in the United States. This feature allows users to stay informed on political developments and changes in their communities.

**Ukraine-Russia War and Israel-Hamas War:**

The website covers the ongoing Ukraine-Russia War and Israel-Hamas War, providing news and updates on these conflicts. The website also includes a section for user feedback on the coverage of these events.

**Ad Feedback:**

The website includes a section for users to provide feedback on the ads displayed on the site, including issues such as ad relevance, technical problems, and other concerns. The website also allows users to submit their feedback and express their opinions on the ads.

**Live TV, Watch, and Listen:**

The website offers live TV, watch, and listen options for users to stay updated with the latest news and events. Users can also sign in to their CNN account to access personalized content and features.

**Redistricting Tracker:**

The website includes a redistricting tracker, which provides information on redistricting efforts and changes in the United States. This feature allows users to stay informed on political developments and changes in their communities.

**Ukraine-Russia War and Israel-Hamas War:**

The website covers the ongoing Ukraine-Russia War and Israel-Hamas War, providing news and updates on these conflicts. The website also includes a section for user feedback on the coverage of these events.

**Ad Feedback:**

The website includes a section for users to provide feedback on the ads displayed on the site, including issues such as ad relevance, technical problems, and other concerns. The website also allows users to submit their feedback and express their opinions on the ads.

**Live TV, Watch, and Listen:**

The website offers live TV, watch, and listen options for users to stay updated with the